# E3SM S2S Soil Moisture Memory & Land Coupling (Weeks 1 to 8)

This notebook analyzes **subseasonal land surface memory and land-atmosphere coupling** across **Weeks 1 to 8** (Days 1–56).

### Scientific Motivation
1. **Soil Moisture Persistence**: Soil moisture memory operates on timescales of weeks to months, serving as a primary driver of subseasonal atmospheric predictability over continents.
2. **Land-Atmosphere Coupling**: During transitional and summer regimes, soil moisture deficits modulate surface latent heat flux and amplify heat extremes (negative $r(\text{SM}, \text{TSA})$).
3. **Weekly Lead Evolution**: Quantifies the degradation of initial soil moisture anomalies from Week 1 to Week 8.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

repo_root = Path.cwd()
while repo_root.parent != repo_root and not (repo_root / "esp_lab").is_dir():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from esp_lab.paths import figure_output_dir
from esp_lab.diagnostics.s2s_core import (
    S2S_WEEKLY_WINDOWS,
    get_weekly_window,
    compute_weekly_anomalies,
)
from esp_lab.diagnostics.s2s_io import (
    DEFAULT_DATA_DIR,
    load_s2s_campaign_weekly,
)

print("S2S Land Coupling Diagnostics Module Loaded.")

## Configuration and Control Panel

In [ ]:
# =============================================================================
# USER CONTROL PANEL — S2S LAND COUPLING & MEMORY (WEEKS 1 TO 8)
# =============================================================================

SM_FIELD = "SOILWATER_10CM"
TEMP_FIELD = "TSA"
COMPONENT = "lnd"
GRID = "180x360_aave"

LEAD_WEEKS = list(range(1, 9))
INIT_YEARS = list(range(1980, 1987))
INIT_MONTHS = [5]  # May initialization (warm season coupling)
MEMBERS = [f"EN{i:02d}" for i in range(10)]

TARGET_CASE = "E3SM-4DEnVarOcn"
CASE_PREFIX = "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_4DEnVarOcn"

FIGURE_ROOT = Path("/global/cfs/cdirs/e3sm/www/zhan391/esp-lab_diag")
FIGURE_OUTDIR = figure_output_dir("s2s_skill", COMPONENT, "weekly_coupling", root=FIGURE_ROOT)
FIGURE_OUTDIR.mkdir(parents=True, exist_ok=True)

print(f"Target Case: {TARGET_CASE}")
print(f"Evaluating Soil Moisture ({SM_FIELD}) and Temperature ({TEMP_FIELD})")
print(f"Figure Output Directory: {FIGURE_OUTDIR}")

## Step 1 — Load Weekly Soil Moisture & Surface Temperature (Weeks 1 to 8)

In [ ]:
%%time
print(f"Loading weekly soil moisture ({SM_FIELD})...")
sm_da = load_s2s_campaign_weekly(
    data_root=DEFAULT_DATA_DIR,
    case_prefix=CASE_PREFIX,
    years=INIT_YEARS,
    init_month=INIT_MONTHS[0],
    members=MEMBERS,
    field=SM_FIELD,
    component=COMPONENT,
    grid=GRID,
    weeks=LEAD_WEEKS,
    verbose=False,
)

print(f"Loading weekly land temperature ({TEMP_FIELD})...")
temp_da = load_s2s_campaign_weekly(
    data_root=DEFAULT_DATA_DIR,
    case_prefix=CASE_PREFIX,
    years=INIT_YEARS,
    init_month=INIT_MONTHS[0],
    members=MEMBERS,
    field=TEMP_FIELD,
    component=COMPONENT,
    grid=GRID,
    weeks=LEAD_WEEKS,
    verbose=False,
)

# Derive weekly anomalies
sm_anom = compute_weekly_anomalies(sm_da.mean("M", skipna=True))
temp_anom = compute_weekly_anomalies(temp_da.mean("M", skipna=True))
print("Anomalies successfully computed.")

## Step 2 — Soil Moisture Persistence (Memory) Across Weeks 1 to 8

Computes the correlation between Week 1 initial soil moisture anomalies and subsequent weekly leads ($W=2..8$):

$$r_{\text{mem}}(W) = \text{corr}\left( \text{SM}'(W=1), \text{SM}'(W) \right)$$

In [ ]:
%%time
# Compute persistence relative to Week 1
sm_w1 = sm_anom.sel(L=1)
sm_w1_prime = sm_w1 - sm_w1.mean("Y", skipna=True)

memory_slices = []
for w in range(1, 9):
    sm_w = sm_anom.sel(L=w)
    sm_w_prime = sm_w - sm_w.mean("Y", skipna=True)
    cov = (sm_w1_prime * sm_w_prime).mean("Y", skipna=True)
    var1 = (sm_w1_prime ** 2).mean("Y", skipna=True)
    varw = (sm_w_prime ** 2).mean("Y", skipna=True)
    denom = np.sqrt(var1 * varw)
    r_mem = xr.where(denom > 1e-12, cov / denom, np.nan)
    memory_slices.append(r_mem)

sm_memory = xr.concat(memory_slices, dim=pd.Index(range(1, 9), name="L"))
print("Soil moisture persistence calculated across Weeks 1 to 8.")

## Step 3 — Soil Moisture–Temperature Coupling Across Weeks 1 to 8

Computes the concurrent subseasonal correlation $r(\text{SM}', \text{TSA}')$ for each lead week ($L=1..8$).

In [ ]:
%%time
coupling_slices = []
for w in range(1, 9):
    s_w = sm_anom.sel(L=w)
    t_w = temp_anom.sel(L=w)
    s_prime = s_w - s_w.mean("Y", skipna=True)
    t_prime = t_w - t_w.mean("Y", skipna=True)
    cov = (s_prime * t_prime).mean("Y", skipna=True)
    denom = np.sqrt((s_prime**2).mean("Y", skipna=True) * (t_prime**2).mean("Y", skipna=True))
    r_coup = xr.where(denom > 1e-12, cov / denom, np.nan)
    coupling_slices.append(r_coup)

sm_temp_coupling = xr.concat(coupling_slices, dim=pd.Index(range(1, 9), name="L"))
print("Concurrent SM-Temperature coupling calculated.")

## Step 4 — Coupling & Memory Evolution Maps (Weeks 1 to 8)

In [ ]:
%%time
fig, axes = plt.subplots(
    nrows=2, ncols=4, figsize=(20, 9),
    subplot_kw={"projection": ccrs.PlateCarree(central_longitude=180)}
)
axes = axes.flatten()
levels = np.linspace(-1, 1, 21)

for idx, w in enumerate(range(1, 9)):
    ax = axes[idx]
    ax.coastlines(linewidth=0.8, color="0.2")
    ax.set_global()
    if w in sm_temp_coupling.L.values:
        cw = sm_temp_coupling.sel(L=w)
        cf = ax.contourf(
            cw.lon, cw.lat, cw,
            levels=levels, cmap="coolwarm", extend="both",
            transform=ccrs.PlateCarree()
        )
    w_def = get_weekly_window(w)
    ax.set_title(w_def.label, fontsize=12, fontweight="bold")

cbar_ax = fig.add_axes([0.25, 0.05, 0.5, 0.025])
cbar = fig.colorbar(cf, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Concurrent Soil Moisture–Temperature Correlation r(SM, TSA)", fontsize=12)

fig.suptitle(
    f"{TARGET_CASE} — Land-Atmosphere Coupling Evolution (Weeks 1–8)",
    fontsize=16, fontweight="bold", y=0.98
)
plt.subplots_adjust(bottom=0.12, top=0.92, hspace=0.15, wspace=0.08)

coupling_fig = FIGURE_OUTDIR / f"{TARGET_CASE}_sm_coupling_w1_w8.png"
plt.savefig(coupling_fig, dpi=200, bbox_inches="tight")
print(f"Figure saved: {coupling_fig}")
plt.show()

## Validation & Integrity Check

In [ ]:
assert "L" in sm_memory.dims, "L dimension missing in sm_memory"
assert len(sm_memory.L) == 8, f"Expected 8 weekly leads, found {len(sm_memory.L)}"
assert "L" in sm_temp_coupling.dims, "L dimension missing in sm_temp_coupling"
assert len(sm_temp_coupling.L) == 8, f"Expected 8 weekly leads, found {len(sm_temp_coupling.L)}"

print("Validation SUCCESS: All 8 land coupling and memory weekly leads verified.")